# Investigating Data Before Building ETL Pipeline
All I'm really doing here is looking at different aspects of the raw data to see what I'm dealing with and discover low hanging fruit for transformation

In [2]:
# load raw data
import pandas as pd
df = pd.read_csv('./data/survey.csv')

In [4]:
# describing annual salary of not employed respondents
print(df.loc[df.Employment == 'Not employed', 'annual_salary_usd'].describe())

count        70.000000
mean      77120.757143
std       63359.371152
min           5.000000
25%       25242.000000
50%       73289.000000
75%      122250.000000
max      250000.000000
Name: annual_salary_usd, dtype: float64


In [3]:
# count of respondents with 0 or na for annual salary
print(df['annual_salary_usd'].isna().sum())

84


In [ ]:
# creating a series of column 'annual_salary_usd' where annual salary < 1000 usd
low = df.loc[df.annual_salary_usd < 1000, 'annual_salary_usd']
# description + top 20 rows
print(low.describe())
print(low.value_counts().head(20))

count    124.000000
mean     244.693548
std      268.419862
min        1.000000
25%       46.500000
50%      107.500000
75%      384.000000
max      999.000000
Name: annual_salary_usd, dtype: float64
annual_salary_usd
10.0     3
1.0      3
575.0    3
64.0     3
98.0     2
383.0    2
5.0      2
20.0     2
93.0     2
102.0    2
77.0     2
7.0      2
3.0      2
35.0     2
56.0     2
133.0    1
223.0    1
29.0     1
800.0    1
18.0     1
Name: count, dtype: int64


In [6]:
# investigating where the salaries considered low for the US would be based
print(df.loc[df.annual_salary_usd.between(1000, 15000), ['annual_salary_usd', 'Country']]
        .sort_values(by='annual_salary_usd'))

      annual_salary_usd     Country
2640             1007.0     Ukraine
2461             1007.0     Ukraine
1313             1011.0   Indonesia
2894             1018.0       Nepal
205              1064.0  Bangladesh
...                 ...         ...
1648            15000.0    Cameroon
3332            15000.0       China
3853            15000.0     Nigeria
546             15000.0      Mexico
1357            15000.0    Zimbabwe

[444 rows x 2 columns]


In [ ]:
print(df['Country'].value_counts(dropna=False).to_string(max_rows))

Country
United States of America                                1089
Germany                                                  441
United Kingdom of Great Britain and Northern Ireland     316
India                                                    262
France                                                   223
Canada                                                   175
Ukraine                                                  161
Brazil                                                   132
Italy                                                    130
Netherlands                                              127
Poland                                                   121
Australia                                                119
Spain                                                    114
Sweden                                                    96
Switzerland                                               78
Czech Republic                                            68
Hungary         

In [14]:
mask = df.annual_salary_usd.between(1000, 15000)

# simple counts (include NaN counts)
print(df.loc[mask, 'Country'].value_counts(dropna=False).to_string())

Country
India                                   97
Ukraine                                 48
Brazil                                  22
Bangladesh                              16
Poland                                  14
Mexico                                  13
Pakistan                                12
Hungary                                 10
Germany                                 10
Turkey                                   9
Indonesia                                9
Iran, Islamic Republic of...             8
Romania                                  8
Colombia                                 8
Viet Nam                                 8
Philippines                              7
Nigeria                                  7
Serbia                                   6
South Africa                             6
Egypt                                    6
Chile                                    6
Malaysia                                 5
Russian Federation                       5
Ken

In [ ]:
# finding suspiciously low salaries in countries where expected salary is generally higher
low = df[df.annual_salary_usd.between(1000, 15000)]
rich = ['United States of America', 'Australia', 'Sweden', 'Denmark',
        'Norway', 'Switzerland', 'Germany', 'United Kingdom of Great Britain and Northern Ireland']
sus = low[low.Country.isin(rich)]
print(len(sus))
print(sus[['Country', 'Currency', 'annual_salary_usd', 'WorkExp', 'Employment']].head(20).to_string())
# print((sus.annual_salary_usd * 12).describe())

23
                       Country                  Currency  annual_salary_usd  WorkExp                                            Employment
410   United States of America  USD United States dollar            12000.0      6.0                                              Employed
602                    Germany         EUR European Euro             8075.0      1.0                                               Student
741                     Sweden        SEK\tSwedish krona             6289.0     13.0                                              Employed
788                    Germany         EUR European Euro            13922.0     25.0  Independent contractor, freelancer, or self-employed
833                  Australia    AUD\tAustralian dollar             6501.0      1.0                                               Student
963                    Germany         EUR European Euro            11601.0      2.0                                              Employed
982                    G

In [ ]:
# viewing types of employment
print(df["Employment"].value_counts())

Employment
Employed                                                4215
Independent contractor, freelancer, or self-employed     589
Not employed                                              90
Student                                                   85
I prefer not to say                                       11
Retired                                                   10
Name: count, dtype: int64


In [8]:
print(df.loc[df.Employment == 'Retired', ['Employment', 'annual_salary_usd']])

     Employment  annual_salary_usd
305     Retired                NaN
372     Retired           100991.0
1213    Retired            34804.0
1699    Retired                NaN
1776    Retired                NaN
1917    Retired                NaN
2433    Retired            72000.0
2913    Retired                NaN
3477    Retired           200000.0
3627    Retired              109.0


In [9]:
print(df.loc[df.Employment == 'Retired'])

      ResponseId                Age  \
305        12392  65 years or older   
372        12849    55-64 years old   
1213       18674  65 years or older   
1699        2293    55-64 years old   
1776       24260  65 years or older   
1917       25917    55-64 years old   
2433       30149  65 years or older   
2913        3509    45-54 years old   
3477       39943  65 years or older   
3627       41005  65 years or older   

                                                EdLevel Employment  WorkExp  \
305   Some college/university study without earning ...    Retired     52.0   
372     Master’s degree (M.A., M.S., M.Eng., MBA, etc.)    Retired     51.0   
1213     Professional degree (JD, MD, Ph.D, Ed.D, etc.)    Retired     45.0   
1699    Master’s degree (M.A., M.S., M.Eng., MBA, etc.)    Retired     30.0   
1776  Some college/university study without earning ...    Retired     25.0   
1917    Master’s degree (M.A., M.S., M.Eng., MBA, etc.)    Retired     35.0   
2433       Bachel

In [3]:
print(df["Age"].describe())
print('__________________________________')
print(df["Age"].value_counts(dropna=False))

count                5000
unique                  7
top       25-34 years old
freq                 1892
Name: Age, dtype: object
__________________________________
Age
25-34 years old      1892
35-44 years old      1578
45-54 years old       664
18-24 years old       545
55-64 years old       270
65 years or older      45
Prefer not to say       6
Name: count, dtype: int64


In [14]:

# print(df["DevType"].value_counts())
print("_______________________________")
print(df.loc[df.DevType == "Student", "annual_salary_usd"])

_______________________________
8       29951.0
117     25000.0
233     34035.0
458       277.0
511     26438.0
816     26000.0
833      6501.0
1592     5580.0
1693    25000.0
1865    12291.0
1879    26000.0
1910     6961.0
1956    50000.0
2098        NaN
2318        NaN
2419        NaN
2437        NaN
2740     8353.0
3258     7777.0
3278        NaN
3515       19.0
3765     3160.0
3781        NaN
3916        NaN
3937     4873.0
4210       52.0
4269    18071.0
4447        NaN
4549    81210.0
4703    19028.0
4781    16242.0
4858        NaN
Name: annual_salary_usd, dtype: float64


In [15]:
print(df.isna().mean().sort_values(ascending=False))
print(df.isna().sum(axis=1).value_counts().sort_index())

DatabaseHaveWorkedWith    0.2104
ICorPM                    0.1204
RemoteWork                0.1106
OrgSize                   0.1060
LanguageHaveWorkedWith    0.0698
Industry                  0.0288
annual_salary_usd         0.0168
WorkExp                   0.0104
YearsCode                 0.0042
EdLevel                   0.0010
ResponseId                0.0000
Age                       0.0000
Employment                0.0000
DevType                   0.0000
Country                   0.0000
Currency                  0.0000
dtype: float64
0    3411
1     726
2     310
3     315
4     135
5      71
6      21
7       8
8       3
Name: count, dtype: int64


In [17]:
print(df["EdLevel"].value_counts())

EdLevel
Bachelor’s degree (B.A., B.S., B.Eng., etc.)                                          2283
Master’s degree (M.A., M.S., M.Eng., MBA, etc.)                                       1464
Some college/university study without earning a degree                                 578
Professional degree (JD, MD, Ph.D, Ed.D, etc.)                                         242
Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)     212
Associate degree (A.A., A.S., etc.)                                                    150
Other (please specify):                                                                 48
Primary/elementary school                                                               18
Name: count, dtype: int64


In [20]:
print(df["WorkExp"].isna().mean())

0.0104


In [22]:
print(df['YearsCode'].isna().mean())
print(df['YearsCode'].describe())

0.0042
count    4979.000000
mean       17.734887
std        10.549221
min         1.000000
25%        10.000000
50%        15.000000
75%        25.000000
max       100.000000
Name: YearsCode, dtype: float64


In [24]:
print(df.loc[df.YearsCode > 50, ['YearsCode', 'Age', 'WorkExp', 'annual_salary_usd']])

      YearsCode                Age  WorkExp  annual_salary_usd
305        52.0  65 years or older     52.0                NaN
1099       60.0  65 years or older     25.0            30000.0
1213       54.0  65 years or older     45.0            34804.0
1579      100.0    35-44 years old    100.0           444449.0
2267       60.0  65 years or older     59.0            48000.0
3481       58.0  65 years or older     50.0           100000.0
3627       57.0  65 years or older     50.0              109.0
4737       56.0  65 years or older     55.0           120000.0
4823       52.0  65 years or older     43.0           116015.0


In [28]:
print(df.loc[df.WorkExp > df.YearsCode, ["WorkExp", "YearsCode", "annual_salary_usd"]])

      WorkExp  YearsCode  annual_salary_usd
14        8.0        5.0            36758.0
51        8.0        7.0            55000.0
99       20.0       16.0           355000.0
153      20.0       11.0           180000.0
154      35.0       30.0            52903.0
...       ...        ...                ...
4908     24.0       10.0            40605.0
4910     30.0       20.0             1200.0
4920      8.0        5.0            31090.0
4943      5.0        3.0            34872.0
4981      4.0        1.0               20.0

[199 rows x 3 columns]


In [5]:


print(df["DevType"].value_counts())
print(df["DevType"].nunique())
print(df["DevType"].str.contains(';').sum())

DevType
Developer, full-stack                            1652
Developer, back-end                               865
Architect, software or solutions                  338
Developer, desktop or enterprise applications     234
Developer, front-end                              234
Developer, mobile                                 171
Developer, embedded applications or devices       169
DevOps engineer or professional                   126
Engineering manager                               118
AI/ML engineer                                     93
Data engineer                                      91
Other (please specify):                            73
Data scientist                                     60
Senior executive (C-suite, VP, etc.)               54
Developer, game or graphics                        43
Cloud infrastructure engineer                      43
Academic researcher                                36
Founder, technology or otherwise                   33
Developer, QA or tes

In [31]:
print(df["OrgSize"].value_counts())

OrgSize
20 to 99 employees                                    961
100 to 499 employees                                  879
Less than 20 employees                                724
10,000 or more employees                              676
1,000 to 4,999 employees                              575
500 to 999 employees                                  294
5,000 to 9,999 employees                              198
Just me - I am a freelancer, sole proprietor, etc.    116
I don’t know                                           47
Name: count, dtype: int64


In [32]:
print(pd.crosstab(df.OrgSize, df.Employment))

Employment                                          Employed  \
OrgSize                                                        
1,000 to 4,999 employees                                 561   
10,000 or more employees                                 661   
100 to 499 employees                                     854   
20 to 99 employees                                       932   
5,000 to 9,999 employees                                 194   
500 to 999 employees                                     285   
I don’t know                                              42   
Just me - I am a freelancer, sole proprietor, etc.        23   
Less than 20 employees                                   660   

Employment                                          Independent contractor, freelancer, or self-employed  \
OrgSize                                                                                                    
1,000 to 4,999 employees                                                       

In [36]:
print(df["ICorPM"].describe())
print(df.ICorPM.isna().mean())
print(df.ICorPM.value_counts())

count                       4398
unique                         2
top       Individual contributor
freq                        3844
Name: ICorPM, dtype: object
0.1204
ICorPM
Individual contributor    3844
People manager             554
Name: count, dtype: int64


In [38]:
print(df.RemoteWork.value_counts())
print(df.RemoteWork.isna().mean())

RemoteWork
Remote                                                                          1554
Hybrid (some remote, leans heavy to in-person)                                   894
Hybrid (some in-person, leans heavy to flexibility)                              819
In-person                                                                        613
Your choice (very flexible, you can come in when you want or just as needed)     567
Name: count, dtype: int64
0.1106


In [40]:
print(df.Industry.value_counts())
print(df.Industry.isna().mean())

Industry
Software Development                          2478
Other:                                         377
Fintech                                        276
Internet, Telecomm or Information Services     218
Healthcare                                     216
Banking/Financial Services                     189
Manufacturing                                  174
Transportation, or Supply Chain                152
Retail and Consumer Services                   150
Government                                     134
Media & Advertising Services                   121
Energy                                         112
Higher Education                               103
Computer Systems Design and Services            92
Insurance                                       64
Name: count, dtype: int64
0.0288


In [43]:
print(df.Country.value_counts())
print(df.Country.nunique())
print("____________")
print(df.Currency.value_counts())

Country
United States of America                                1089
Germany                                                  441
United Kingdom of Great Britain and Northern Ireland     316
India                                                    262
France                                                   223
                                                        ... 
Angola                                                     1
Maldives                                                   1
Myanmar                                                    1
Saudi Arabia                                               1
Jordan                                                     1
Name: count, Length: 130, dtype: int64
130
____________
Currency
EUR European Euro           1460
USD United States dollar    1323
GBP Pound sterling           315
INR Indian rupee             254
CAD Canadian dollar          172
                            ... 
UYU\tUruguayan peso            1
MVR\tMaldivian rufiyaa   

In [42]:
print(pd.crosstab(df.Country, df.Currency))

Currency                              AED United Arab Emirates dirham  \
Country                                                                 
Albania                                                             0   
Algeria                                                             0   
Angola                                                              0   
Argentina                                                           0   
Armenia                                                             0   
...                                                               ...   
Uzbekistan                                                          0   
Venezuela, Bolivarian Republic of...                                0   
Viet Nam                                                            0   
Yemen                                                               0   
Zimbabwe                                                            0   

Currency                              ALL\tAlbania

In [46]:
print(df.Country.value_counts().head(30))

Country
United States of America                                1089
Germany                                                  441
United Kingdom of Great Britain and Northern Ireland     316
India                                                    262
France                                                   223
Canada                                                   175
Ukraine                                                  161
Brazil                                                   132
Italy                                                    130
Netherlands                                              127
Poland                                                   121
Australia                                                119
Spain                                                    114
Sweden                                                    96
Switzerland                                               78
Czech Republic                                            68
Hungary         